# Real paired multimodal

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
exp = "pbmc_paired"
result_name = "real_paired"
results_path = f"/home/mila/m/myriam.lizotte/scratch/RF-MALI/results/{exp}"

# load results
results_df = pd.read_csv(f"{results_path}/{result_name}_results.csv")

# in the df, replace "_" with " " 
results_df.columns = results_df.columns.str.replace('_', ' ')

# # convert relevant columns to numeric
num_cols = results_df.columns.difference(['model'])
# results_df[num_cols] = results_df[num_cols].apply(pd.to_numeric, errors='coerce')


# take the abs for silhouette domain because it should be low, doesnt matter the sign
results_df['Silhouette domain'] = results_df['Silhouette domain'].abs()
results_df.index = results_df['model']
results_df.drop(columns=['model'], inplace=True)
results_df

,seed,t,FOSCTTM,Silhouette domain,Accuracy missing,Accuracy visible,Alignment score
model,,,,,,,
FoSTA,690349,2,0.146160,0.009679,0.754172,0.904365,0.703021
RFMALI,690349,2,0.151174,0.007101,0.672657,0.864570,0.729140
MALI,690349,2,0.087519,0.000218,0.896662,0.917843,0.992127
Pamona,690349,2,0.455036,0.004868,0.228498,0.292683,0.355051
KEMArbf,690349,2,0.437607,0.036165,0.247754,0.413350,0.794774
KEMAlin,690349,2,0.380895,0.079097,0.280488,0.403081,0.374400
FoSTA,8925503,2,0.147417,0.009733,0.745828,0.888960,0.704325
RFMALI,8925503,2,0.148843,0.007170,0.663671,0.852375,0.719403
MALI,8925503,2,0.083755,0.000212,0.893453,0.911425,0.995341


average over all datasets

# print it to a table

In [3]:
# clean summary_df 
summary_df = results_df.groupby(results_df.index).mean()
# remove "RFMALI" 
summary_df = summary_df.loc[summary_df.index != 'RFMALI']

cols_to_keep = ["Accuracy missing", "Alignment score", "FOSCTTM"]
# rename Accuracy missing to Accuracy
summary_df = summary_df[cols_to_keep]
summary_df = summary_df.rename(columns={"Accuracy missing": "Accuracy"})

In [4]:
# Function to format the top 3 values: bold, underline, italic (adjusted for lower is better columns)
def format_top_3(df):
    formatted_df = df.copy()
    
    # List of columns where lower values are better
    lower_is_better_columns = ['FOSCTTM', 'Silhouette domain']
    
    
    for column in df.columns:  # Start from the first numerical column
        if column in lower_is_better_columns:
            # Find the smallest 3 values (lower is better)
            top_3 = df[column].nsmallest(3).values
        else:
            # Find the largest 3 values (higher is better)
            top_3 = df[column].nlargest(3).values
        
        if len(top_3) >= 3:
            # Apply formatting: bold for best, underline for second, italic for third
            formatted_df.loc[df.index, column] = df[column].apply(
                lambda x: f"\\textbf{{{x:.3f}}}" if x == top_3[0] else 
                            (f"\\underline{{{x:.3f}}}" if x == top_3[1] else 
                            (f"\\textit{{{x:.3f}}}" if x == top_3[2] else f"{x:.3f}"))
            )
    
    return formatted_df

In [5]:
format_top_3(summary_df)

/tmp/ipykernel_3762446/2014713946.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['\\underline{0.742}' '0.243' '\\textit{0.247}' '\\textbf{0.901}' '0.237']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  formatted_df.loc[df.index, column] = df[column].apply(
/tmp/ipykernel_3762446/2014713946.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['\\textit{0.702}' '0.356' '\\underline{0.809}' '\\textbf{0.993}' '0.356']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  formatted_df.loc[df.index, column] = df[column].apply(
/tmp/ipykernel_3762446/2014713946.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['\\underline{0.146}' '\\textit{0.399}' '0.425

,Accuracy,Alignment score,FOSCTTM
model,,,
FoSTA,\underline{0.742},\textit{0.702},\underline{0.146}
KEMAlin,0.243,0.356,\textit{0.399}
KEMArbf,\textit{0.247},\underline{0.809},0.425
MALI,\textbf{0.901},\textbf{0.993},\textbf{0.085}
Pamona,0.237,0.356,0.455


In [6]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(format_top_3(summary_df).to_latex(
        escape=False,
        multirow=True,
        float_format="%.3f"
    )) 

# then copy the output latex table into a .tex file for inclusion in the paper

\begin{tabular}{llll}
\toprule
 & Accuracy & Alignment score & FOSCTTM \\
model &  &  &  \\
\midrule
FoSTA & \underline{0.742} & \textit{0.702} & \underline{0.146} \\
KEMAlin & 0.243 & 0.356 & \textit{0.399} \\
KEMArbf & \textit{0.247} & \underline{0.809} & 0.425 \\
MALI & \textbf{0.901} & \textbf{0.993} & \textbf{0.085} \\
Pamona & 0.237 & 0.356 & 0.455 \\
\bottomrule
\end{tabular}

